In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [ ]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.neural_network import MLPClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

In [ ]:
train = pd.read_csv("/kaggle/input/competitions/outfit-prediction-c-1-fall-26-27/fashion_train.csv")
test = pd.read_csv("/kaggle/input/competitions/outfit-prediction-c-1-fall-26-27/fashion_test.csv")

In [ ]:
X = train.drop(columns=["Id","target"])
y = train["target"]

In [ ]:
X_test = test.drop(columns=["Id"])

In [ ]:
X_train, X_valid, y_train, y_valid = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

In [ ]:
pipeline = Pipeline([
    ("scaler", MinMaxScaler()),
    ("mlp", MLPClassifier(
        random_state=42,
        early_stopping=True
    ))
])

In [ ]:
param_grid = {
    "mlp__hidden_layer_sizes":[
        (128,),
        (256,),
        (128,64),
        (256,128)
    ],
    "mlp__activation":[
        "relu",
        "tanh"
    ],
    "mlp__alpha":[
        0.0001,
        0.001,
        0.01
    ],
    "mlp__learning_rate_init":[
        0.001,
        0.0005
    ],
    "mlp__batch_size":[
        64,
        128
    ],
    "mlp__max_iter":[100]
}

In [ ]:
grid = GridSearchCV(
    pipeline,
    param_grid,
    cv=3,
    scoring="accuracy",
    n_jobs=-1,
    verbose=2
)

In [ ]:
grid.fit(X_train, y_train)

In [ ]:
print("Best Parameters:")
print(grid.best_params_)

In [ ]:
print("Best CV Accuracy:")
print(grid.best_score_)

In [ ]:
best_model = grid.best_estimator_

In [ ]:
pred = best_model.predict(X_valid)

In [ ]:
print("Validation Accuracy:", accuracy_score(y_valid,pred))

In [ ]:
print(classification_report(y_valid,pred))

In [ ]:
print(confusion_matrix(y_valid,pred))

In [ ]:
best_model.fit(X, y)

In [ ]:
test_pred = best_model.predict(X_test)

In [ ]:
submission = pd.DataFrame({
    "Id": test["Id"],
    "target": test_pred
})

In [ ]:
submission.to_csv("submission.csv", index=False)

In [ ]:
print(submission.head())